#### RQ1: Can consistent dimensional representations of emotions be extracted from vision–language models (VLMs)?

We use [Intraclass Correlation Coefficient](https://en.wikipedia.org/wiki/Intraclass_correlation) to measure the consistency of the dimensional representations.

In [13]:
import pandas as pd, numpy as np, ast
from scipy.stats import ttest_ind
from pathlib import Path

# --- Config ---
BASE_DIR = Path.cwd()
RESULTS_PATH = BASE_DIR.parent / "outputs" / "results" / "processed" / "semantic" / "emotion_dimension_summary.csv"
OUTPUT_PATH = BASE_DIR.parent / "outputs" / "results" / "statistics" / "definition_confidence_summary.csv"

# --- Load data ---
df = pd.read_csv(RESULTS_PATH)
print(f"[INFO] Loaded {len(df)} rows from {RESULTS_PATH}")

# --- Parse lists ---
for col in df.columns:
    if df[col].astype(str).str.startswith("[").any():
        df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith("[") else x)

# --- Detect dimensions ---
exclude = {"model","emotion","definition","client"}
dim_cols = [c for c in df.columns if c not in exclude and "mean" not in c.lower()]
print(f"[INFO] Dimensions: {dim_cols}")

# --- Compute definition sensitivity + confidence ---
results=[]
for (model, emotion), g in df.groupby(["model", "emotion"]):
    defs = g["definition"].unique()
    if len(defs) != 2: 
        continue
    for dim in dim_cols:
        v1 = np.array(g[g["definition"]==defs[0]][dim].values[0],dtype=float)
        v2 = np.array(g[g["definition"]==defs[1]][dim].values[0],dtype=float)
        if len(v1)<2 or len(v2)<2: 
            continue
        if np.allclose(v1, v2) or v1.std()==0 and v2.std()==0:
            p = 1.0  # perfectly consistent
        else:
            stat, p = ttest_ind(v1, v2, equal_var=False)
        results.append({
            "model":model,"emotion":emotion,"dimension":dim,
            "definition_pval":p,
            "std_cambridge":v1.std(),"std_sentiwordnet":v2.std(),
            "mean_cambridge":v1.mean(),"mean_sentiwordnet":v2.mean()
        })

out=pd.DataFrame(results)
out["significant"] = out["definition_pval"] < 0.05
conf_limit = 0.3
# summary per model
summary = out.groupby("model")["significant"].agg(["sum","count"])
summary["percent_significant"] = (summary["sum"] / summary["count"] * 100).round(1)
print(f"{'='*50}\nSignificant differences between definitions:")
print(summary)

# add coefficient of variation (CV) and confidence flag
out["cv_cambridge"] = (out["std_cambridge"] / out["mean_cambridge"].abs()).round(4)
out["cv_sentiwordnet"] = (out["std_sentiwordnet"] / out["mean_sentiwordnet"].abs()).round(4)
out["confident"] = (out[["cv_cambridge","cv_sentiwordnet"]] < conf_limit).all(axis=1)

out["conf_cv_sentiwordnet"] = (out[["cv_sentiwordnet"]] < conf_limit).all(axis=1)
out["conf_cv_cambridge"] = (out[["cv_cambridge"]] < conf_limit).all(axis=1)

# summary per model
conf_summary = out.groupby("model")["conf_cv_sentiwordnet"].agg(["sum","count"])
conf_summary["percent_confident"] = (conf_summary["sum"] / conf_summary["count"] * 100).round(1)
print(f"{'='*50}\nCoefficient of Variation < {conf_limit} for SentiWordNet:")
print(conf_summary)

# summary per model
conf_summary = out.groupby("model")["conf_cv_cambridge"].agg(["sum","count"])
conf_summary["percent_confident"] = (conf_summary["sum"] / conf_summary["count"] * 100).round(1)
print(f"{'='*50}\nCoefficient of Variation < {conf_limit} for Cambridge:")
print(conf_summary)

# summary per model
conf_summary = out.groupby("model")["confident"].agg(["sum","count"])
conf_summary["percent_confident"] = (conf_summary["sum"] / conf_summary["count"] * 100).round(1)
print(f"{'='*50}\nCoefficient of Variation < {conf_limit} for both definitions:")
print(conf_summary)

OUTPUT_PATH.parent.mkdir(parents=True,exist_ok=True)
out.to_csv(OUTPUT_PATH,index=False)
print(f"[✅ Saved results] {OUTPUT_PATH}")
display(out.head())


[INFO] Loaded 192 rows from /home/kai/Repositories/commonsense_kg_construction/outputs/results/processed/semantic/emotion_dimension_summary.csv
[INFO] Dimensions: ['activity', 'arousal', 'dominance', 'evaluation', 'pleasure', 'potency', 'unpredictability', 'valence']


/home/kai/Repositories/commonsense_kg_construction/.venv/lib/python3.10/site-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


Significant differences between definitions:
                                         sum  count  percent_significant
model                                                                   
groq_llama-4-maverick-17b-128e-instruct  118    256                 46.1
groq_llama-4-scout-17b-16e-instruct      111    256                 43.4
openai_gpt-5                             145    256                 56.6
Coefficient of Variation < 0.3 for SentiWordNet:
                                         sum  count  percent_confident
model                                                                 
groq_llama-4-maverick-17b-128e-instruct  247    256               96.5
groq_llama-4-scout-17b-16e-instruct      248    256               96.9
openai_gpt-5                             243    256               94.9
Coefficient of Variation < 0.3 for Cambridge:
                                         sum  count  percent_confident
model                                                                

,model,emotion,dimension,definition_pval,std_cambridge,std_sentiwordnet,mean_cambridge,mean_sentiwordnet,significant,cv_cambridge,cv_sentiwordnet,confident,conf_cv_sentiwordnet,conf_cv_cambridge
0,groq_llama-4-maverick-17b-128e-instruct,acceptance,activity,2.194331e-12,1.561249e-02,4.582576e-02,0.4025,0.3300,True,0.0388,0.1389,True,True,True
1,groq_llama-4-maverick-17b-128e-instruct,acceptance,arousal,2.254705e-21,0.000000e+00,4.993746e-02,0.4000,0.2475,True,0.0000,0.2018,True,True,True
2,groq_llama-4-maverick-17b-128e-instruct,acceptance,dominance,1.000000e+00,2.775558e-17,2.775558e-17,0.2000,0.2000,False,0.0000,0.0000,True,True,True
3,groq_llama-4-maverick-17b-128e-instruct,acceptance,evaluation,1.302046e-29,0.000000e+00,4.974937e-02,0.5000,0.7550,True,0.0000,0.0659,True,True,True
4,groq_llama-4-maverick-17b-128e-instruct,acceptance,pleasure,5.990444e-11,0.000000e+00,6.000000e-02,0.5000,0.6800,True,0.0000,0.0882,True,True,True


In [ ]:
import pandas as pd, numpy as np, ast
from scipy.stats import ttest_ind
from pathlib import Path

# --- Config ---
BASE_DIR = Path.cwd()
RESULTS_PATH = BASE_DIR.parent / "outputs" / "results" / "processed" / "semantic" / "emotion_dimension_summary.csv"
OUTPUT_PATH = BASE_DIR.parent / "outputs" / "results" / "statistics" / "definition_confidence_summary.csv"

# --- Load data ---
df = pd.read_csv(RESULTS_PATH)
print(f"[INFO] Loaded {len(df)} rows from {RESULTS_PATH}")

# --- Parse lists ---
for col in df.columns:
    if df[col].astype(str).str.startswith("[").any():
        df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith("[") else x)

# --- Detect dimensions ---
exclude = {"model","emotion","definition","client"}
dim_cols = [c for c in df.columns if c not in exclude and "mean" not in c.lower()]
print(f"[INFO] Dimensions: {dim_cols}")

dim_ranges = {dim: df[dim].apply(lambda x: max(ast.literal_eval(x)) - min(ast.literal_eval(x))
             if isinstance(x,str) and x.startswith("[") else np.nan).mean() for dim in dim_cols}

results=[]
for (model, emotion), g in df.groupby(["model", "emotion"]):
    defs=g["definition"].unique()
    if len(defs)!=2: continue
    for dim in dim_cols:
        v1=np.array(g[g["definition"]==defs[0]][dim].values[0],float)
        v2=np.array(g[g["definition"]==defs[1]][dim].values[0],float)
        if len(v1)<2 or len(v2)<2: continue
        rng=dim_ranges[dim] if dim_ranges[dim]>0 else 1
        stat,p=ttest_ind(v1,v2,equal_var=False)
        results.append({
            "model":model,"emotion":emotion,"dimension":dim,"definition_pval":p,
            "std_cambridge":v1.std()/rng,"std_sentiwordnet":v2.std()/rng,
            "mean_cambridge":v1.mean(),"mean_sentiwordnet":v2.mean()
        })
out=pd.DataFrame(results)
out["significant"] = out["definition_pval"] < 0.05
conf_limit = 0.1
# summary per model
summary = out.groupby("model")["significant"].agg(["sum","count"])
summary["percent_significant"] = (summary["sum"] / summary["count"] * 100).round(1)
print(f"{'='*50}\nSignificant differences between definitions:")
print(summary)

# add coefficient of variation (CV) and confidence flag
out["cv_cambridge"] = (out["std_cambridge"] / out["mean_cambridge"].abs()).round(4)
out["cv_sentiwordnet"] = (out["std_sentiwordnet"] / out["mean_sentiwordnet"].abs()).round(4)
out["confident"] = (out[["cv_cambridge","cv_sentiwordnet"]] < conf_limit).all(axis=1)

out["conf_cv_sentiwordnet"] = (out[["cv_sentiwordnet"]] < conf_limit).all(axis=1)
out["conf_cv_cambridge"] = (out[["cv_cambridge"]] < conf_limit).all(axis=1)

# summary per model
conf_summary = out.groupby("model")["conf_cv_sentiwordnet"].agg(["sum","count"])
conf_summary["percent_confident"] = (conf_summary["sum"] / conf_summary["count"] * 100).round(1)
print(f"{'='*50}\nCoefficient of Variation < {conf_limit} for SentiWordNet:")
print(conf_summary)

# summary per model
conf_summary = out.groupby("model")["conf_cv_cambridge"].agg(["sum","count"])
conf_summary["percent_confident"] = (conf_summary["sum"] / conf_summary["count"] * 100).round(1)
print(f"{'='*50}\nCoefficient of Variation < {conf_limit} for Cambridge:")
print(conf_summary)

# summary per model
conf_summary = out.groupby("model")["confident"].agg(["sum","count"])
conf_summary["percent_confident"] = (conf_summary["sum"] / conf_summary["count"] * 100).round(1)
print(f"{'='*50}\nCoefficient of Variation < {conf_limit} for both definitions:")
print(conf_summary)

print(f"{'='*50}\nAbsolute STD < {conf_limit} for both definitions:")
out=pd.DataFrame(results)
out["significant"]=out["definition_pval"]<0.05
conf_limit=0.2
out["conf_cambridge"]=out["std_cambridge"]<conf_limit
out["conf_sentiwordnet"]=out["std_sentiwordnet"]<conf_limit
out["conf_both"]=out["conf_cambridge"]&out["conf_sentiwordnet"]
print(f"{'='*50}\nAbsolute STD < {conf_limit} for both definitions:")
print(out.groupby("model")["significant"].agg(["sum","count"]))
print(out.groupby("model")["conf_both"].agg(["sum","count"]))
sig_summary = out.groupby("model")["significant"].agg(["sum","count"])
sig_summary["ratio"] = (sig_summary["sum"] / sig_summary["count"]).round(3)*100
conf_summary = out.groupby("model")["conf_both"].agg(["sum","count"])
conf_summary["ratio"] = (conf_summary["sum"] / conf_summary["count"]).round(3)*100
print(f"{'='*50}\nAbsolute STD < {conf_limit} for both definitions:")
print(sig_summary)
print(conf_summary)
OUTPUT_PATH.parent.mkdir(parents=True,exist_ok=True)
out.to_csv(OUTPUT_PATH,index=False)
print(f"[✅ Saved results] {OUTPUT_PATH}")
display(out.head())


[INFO] Loaded 192 rows from /home/kai/Repositories/commonsense_kg_construction/outputs/results/processed/semantic/emotion_dimension_summary.csv
[INFO] Dimensions: ['activity', 'arousal', 'dominance', 'evaluation', 'pleasure', 'potency', 'unpredictability', 'valence']


/home/kai/Repositories/commonsense_kg_construction/.venv/lib/python3.10/site-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


Significant differences between definitions:
                                         sum  count  percent_significant
model                                                                   
groq_llama-4-maverick-17b-128e-instruct  118    256                 46.1
groq_llama-4-scout-17b-16e-instruct      123    256                 48.0
openai_gpt-5                             145    256                 56.6
Coefficient of Variation < 0.1 for SentiWordNet:
                                         sum  count  percent_confident
model                                                                 
groq_llama-4-maverick-17b-128e-instruct  217    256               84.8
groq_llama-4-scout-17b-16e-instruct      228    256               89.1
openai_gpt-5                             168    256               65.6
Coefficient of Variation < 0.1 for Cambridge:
                                         sum  count  percent_confident
model                                                                

,model,emotion,dimension,definition_pval,std_cambridge,std_sentiwordnet,mean_cambridge,mean_sentiwordnet,significant,conf_cambridge,conf_sentiwordnet,conf_both
0,groq_llama-4-maverick-17b-128e-instruct,acceptance,activity,2.194331e-12,1.561249e-02,4.582576e-02,0.4025,0.3300,True,True,True,True
1,groq_llama-4-maverick-17b-128e-instruct,acceptance,arousal,2.254705e-21,0.000000e+00,4.993746e-02,0.4000,0.2475,True,True,True,True
2,groq_llama-4-maverick-17b-128e-instruct,acceptance,dominance,1.000000e+00,2.775558e-17,2.775558e-17,0.2000,0.2000,False,True,True,True
3,groq_llama-4-maverick-17b-128e-instruct,acceptance,evaluation,1.302046e-29,0.000000e+00,4.974937e-02,0.5000,0.7550,True,True,True,True
4,groq_llama-4-maverick-17b-128e-instruct,acceptance,pleasure,5.990444e-11,0.000000e+00,6.000000e-02,0.5000,0.6800,True,True,True,True
